# Étape 2 & 3 — Entraînement des modèles et tracking MLflow

Chargement du dataset préparé à l'Étape 1, puis entraînement de modèles de scoring
avec suivi des expérimentations via MLflow.

In [1]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

In [2]:
# On recharge le dataset propre de l'Étape 1 (instantané grâce au Parquet)
df = pd.read_parquet("../data/df_prepared.parquet")

# X = les variables explicatives ; y = la cible
# On retire TARGET (la réponse) et SK_ID_CURR (un simple identifiant, sans valeur prédictive)
X = df.drop(columns=["TARGET", "SK_ID_CURR"])
y = df["TARGET"]

print("X :", X.shape, "| y :", y.shape)
print("Répartition de y :", y.value_counts(normalize=True).round(3).to_dict())

X : (307511, 800) | y : (307511,)
Répartition de y : {0: 0.919, 1: 0.081}


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train :", X_train.shape, "| Test :", X_test.shape)
print("Taux de défaut — train :", round(y_train.mean(), 4), "| test :", round(y_test.mean(), 4))


Train : (246008, 800) | Test : (61503, 800)
Taux de défaut — train : 0.0807 | test : 0.0807


In [4]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")   # base à la racine du projet
mlflow.set_experiment("scoring_credit")

2026/07/07 13:26:13 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/07 13:26:13 INFO mlflow.store.db.utils: Updating database tables
2026/07/07 13:26:14 INFO mlflow.tracking.fluent: Experiment with name 'scoring_credit' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/matthieu/dev/openclassrooms/projet6/notebooks/mlruns/1', creation_time=1783423574127, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783423574127, lifecycle_stage='active', name='scoring_credit', tags={}, trace_location=None, workspace='default'>

In [5]:
with mlflow.start_run(run_name="baseline_histgb"):
    params = {"max_iter": 200, "learning_rate": 0.1}
    model = HistGradientBoostingClassifier(**params, random_state=42)
    model.fit(X_train, y_train)

    # Prédictions sur le test
    proba = model.predict_proba(X_test)[:, 1]   # probabilité de défaut
    auc = roc_auc_score(y_test, proba)
    acc = accuracy_score(y_test, model.predict(X_test))

    # On enregistre TOUT dans MLflow
    mlflow.log_params(params)
    mlflow.log_metric("AUC", auc)
    mlflow.log_metric("accuracy", acc)
    mlflow.sklearn.log_model(model, artifact_path="model")

    print(f"AUC = {auc:.4f} | accuracy = {acc:.4f}")

2026/07/07 13:26:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/07 13:26:56 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


AUC = 0.7823 | accuracy = 0.9202
